# Qismet: 16-group scaling experiment

This scaling run did not establish a quantum speed advantage. Greedy swaps and simulated annealing found an exact QUBO optimum in all six cases.

## What ran

Fashion-MNIST, the existing 5,000 training/1,000 validation/10,000 test split; fully connected network **784→32→10**. There are **16 groups of two neurons** and **16 logical qubits**. Three fresh initializations: 1001, 2002, 3003. The training budget is 20 epochs per mask, unchanged from the smaller experiment.

At 50% pruning, retain eight groups (16 neurons): **12,870** masks and **12,730 parameters**, versus **25,450** dense. At 75%, retain four groups (eight neurons): **1,820** masks and **6,370 parameters**.

For each initialization and pruning budget, 192 distinct masks fit the surrogate and 64 disjoint masks audit it. Total **1,536 retraining labels**. Linear, full-quadratic and ring-quadratic surrogates are audited; the full quadratic is the predeclared primary objective. Ridge regularization uses only cross-validation on the fit masks. All QAOA and classical selections are frozen before their final test evaluation.

## Local comparisons

QAOA here means an **ideal subspace simulation**, independently checked against the actual Qiskit circuit on the hardware initialization. One and two layers, three fixed optimizer starts per depth, then depth selection by predicted mean QUBO cost. It uses a number-preserving ring XY mixer and an initial state containing exactly the required number of retained groups.

| Method | 50% pruning: mean test accuracy | 75% pruning: mean test accuracy |
|---|---:|---:|
| QAOA best32 | 82.86% | 81.05% |
| Random best32 | 82.60% | 81.49% |
| Exact linear | 82.73% | 81.52% |
| Greedy swaps | 82.92% | 81.10% |
| Simulated annealing | 82.92% | 81.10% |
| Exact QUBO | 82.92% | 81.10% |


These are actual selected-network test accuracies, averaged over three initializations. One frozen 32-candidate draw per method per initialization is used, so differences include sampling variation. They are not the expected-distribution metrics reported in the earlier eight-group experiment. All individual cases remain in `all_comparisons.csv`.

![Local results](local_performance.png)

## Approximation quality

The full quadratic's mean held-out R² is **0.591** at 50% pruning and **0.625** at 75%. Mean absolute validation-loss error is **0.0054** and **0.0221**. Linear and sparse-ring alternatives are reported in `all_surrogate_audits.csv`. A better QUBO minimum does not guarantee better true retraining performance, because this remains an approximation.

## Real 16-qubit hardware results

Four jobs ran on **ibm_fez**: an initial-state control and one-layer QAOA at each pruning budget, using 4,096 shots each. Reported QPU usage for this follow-up totals **12 seconds**; completed-job reported usage across the project is **44 seconds**. Prior uncertain usage remains reserved separately. Two-layer circuits were simulated but were not submitted as extra hardware jobs.

| Pruning | Q valid shots | Mean Q accuracy, successful batches | Mean random accuracy | Failed Q batches |
|---|---:|---:|---:|---:|
| 50% | 28.5% | 82.65% | 82.72% | 0/128 |
| 75% | 25.1% | 81.23% | 81.36% | 0/128 |


The aggregate hardware counts were shuffled with a fixed seed and partitioned into **128 reconstructed 32-shot selections**. Each quantum selection chooses its best feasible mask by QUBO score; each random selection has 32 feasible candidates scored by the same QUBO. All winning states were frozen before their networks were retrained and tested. These are partitions of **one measurement pool**, not 128 independent quantum jobs. No device-repeat confidence interval is claimed.

Invalid quantum outcomes consume the fixed 32-shot budget. Failed batches are reported explicitly. The successful-batch accuracy in the table excludes failed batches; `hardware/keep_*/summary.json` also reports the utility when an unusable batch receives zero. The initial-state controls and every raw outcome are retained.

![Hardware results](hardware_performance.png)

## Search times and total work

| Classical search | 50% pruning: median seconds | 75% pruning: median seconds |
|---|---:|---:|
| Greedy swaps | 0.003804 | 0.002922 |
| Simulated annealing | 0.021901 | 0.021635 |
| Exact QUBO, enumerate and score | 0.049465 | 0.006640 |


These direct classical baselines score proposed masks from coefficients; greedy and annealing do not access a precomputed exhaustive score table. The primary local simulation uses an exhaustive energy table for state evolution; its earlier cached-table baseline timings are retained in per-run metadata but are not used in the table above.

Classical exact timings include enumeration and feature construction, score evaluation and argmin. They are measured implementations, not lower bounds on optimized classical performance. QPU-reported usage is a different clock from CPU elapsed time. Each hardware job's queue-inclusive observed time is in the ledger. `all_qaoa_optimization.csv` records simulator angle-optimization time and evaluations. `preparation_times.csv` records label training and regression, including audit/ablation work. All solvers share access to the learned surrogate.

Increasing the search space did not force classical methods to enumerate every candidate: local heuristics can exploit the QUBO structure. This study therefore does not establish quantum advantage or an end-to-end speedup. The meaningful outcomes are the measured pruning quality, approximation quality, solver gaps and hardware acceptance at the larger size.

## Code and evidence

- `experiment.py`: neural network, initialization rewind, fit/audit labels, surrogate fitting, ideal QAOA, classical comparisons.
- `hardware.py`: actual 16-qubit circuit, independent equivalence checks, IBM submissions and inherited usage accounting.
- `direct_classical.py`: direct-coefficient greedy, annealing and exact-search timings.
- `evaluate_hardware.py`: frozen 32-shot comparisons and final network evaluation.
- `hardware/ledger.json`: actual job IDs, device, counts paths, usage and elapsed-time evidence.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
BASE=Path('.').resolve()
display(pd.read_csv(BASE/'all_comparisons.csv')[['seed','pruning','method','parameters','test_accuracy','validation_drop_pp','qubo_gap']])
display(pd.read_csv(BASE/'all_surrogate_audits.csv'))
display(pd.read_csv(BASE/'all_direct_search_times.csv'))


,seed,pruning,method,parameters,test_accuracy,validation_drop_pp,qubo_gap
0,1001,50.0,QAOA best32,12730,0.8291,-0.600004,2.616607e-03
1,1001,50.0,Random best32,12730,0.8236,-1.000005,4.098621e-03
2,1001,50.0,Exact linear,12730,0.8262,-1.100004,5.822041e-03
3,1001,50.0,Exact QUBO,12730,0.8273,-0.300002,0.000000e+00
4,1001,50.0,Greedy swaps,12730,0.8273,-0.300002,0.000000e+00
5,1001,50.0,Simulated annealing,12730,0.8273,-0.300002,0.000000e+00
6,1001,75.0,QAOA best32,6370,0.8108,2.199996,0.000000e+00
7,1001,75.0,Random best32,6370,0.8180,0.699997,2.708147e-02
8,1001,75.0,Exact linear,6370,0.8167,0.199997,8.658591e-03
9,1001,75.0,Exact QUBO,6370,0.8108,2.199996,0.000000e+00


,seed,pruning,model,alpha,r2,mae,spearman
0,1001,50.0,Linear,3.162278,0.649478,0.004309,0.807372
1,1001,50.0,Quadratic,10.000000,0.700739,0.004080,0.853114
2,1001,50.0,Ring quadratic,3.162278,0.587435,0.004871,0.767720
3,1001,75.0,Linear,1.000000,0.676388,0.023113,0.889194
4,1001,75.0,Quadratic,3.162278,0.785777,0.017804,0.903205
5,1001,75.0,Ring quadratic,1.000000,0.668428,0.023787,0.882051
6,2002,50.0,Linear,3.162278,0.497437,0.005574,0.685668
7,2002,50.0,Quadratic,31.622777,0.489409,0.005622,0.683242
8,2002,50.0,Ring quadratic,3.162278,0.502929,0.005656,0.698352
9,2002,75.0,Linear,3.162278,0.254102,0.026276,0.701374


,seed,pruning,method,seconds,clock
0,1001,50.0,Greedy swaps,0.004032,"CPU elapsed, no cached score table"
1,1001,50.0,Simulated annealing,0.023475,"CPU elapsed, no cached score table"
2,1001,50.0,"Exact QUBO, enumerate and score",0.049465,"CPU elapsed, no cached score table"
3,1001,75.0,Greedy swaps,0.002922,"CPU elapsed, no cached score table"
4,1001,75.0,Simulated annealing,0.021388,"CPU elapsed, no cached score table"
5,1001,75.0,"Exact QUBO, enumerate and score",0.006640,"CPU elapsed, no cached score table"
6,2002,50.0,Greedy swaps,0.003236,"CPU elapsed, no cached score table"
7,2002,50.0,Simulated annealing,0.021901,"CPU elapsed, no cached score table"
8,2002,50.0,"Exact QUBO, enumerate and score",0.052282,"CPU elapsed, no cached score table"
9,2002,75.0,Greedy swaps,0.003535,"CPU elapsed, no cached score table"


## Hardware outcomes and actual circuit source

In [2]:
import inspect
import hardware
print(inspect.getsource(hardware.build))
display(pd.read_csv(BASE/'hardware'/'all_distribution_metrics.csv'))
display(pd.read_csv(BASE/'hardware'/'all_accuracy_comparisons.csv'))


def build(keep,layers=1):
    a=np.load(HERE/'seed_1001'/f'keep_{keep}'/f'qaoa_p{max(layers,1)}.npz')
    qc=QuantumCircuit(16,name=f'scale16_keep{keep}_p{layers}')
    block=16//keep
    for start in range(0,16,block):
        if block==2:
            qc.h(start); qc.x(start+1); qc.cx(start,start+1)
        else:
            # Uniform single excitation in each four-neuron-group block.
            qc.x(start)
            for i in range(block-1):
                angle=2*np.arccos(1/np.sqrt(block-i))
                qc.cry(angle,start+i,start+i+1); qc.cx(start+i+1,start+i)
    if layers==0: return qc
    linear=a['linear']; b=np.zeros((16,16)); b[np.triu_indices(16,1)]=a['pairs']
    h=-linear/2-(b+b.T).sum(1)/4; J=b/4; theta=a['theta']; scale=float(a['scale'])
    for l in range(layers):
        for i in range(16): qc.rz(2*theta[l]*h[i]/scale,i)
        for i in range(16):
            for j in range(i+1,16): qc.rzz(2*theta[l]*J[i,j]/scale,i,j)
        for i,j in e.EDGES:
            qc.

,keep,layer,job_id,qpu_seconds,shots,feasible_fraction,mean_feasible_qubo,uniform_mean_qubo,top10_probability_conditional,top10_probability_raw,uniform_top10_probability
0,8,0,dbmjbdl8ft8s738uv72g,3.0,4096,0.743652,0.529959,0.530133,0.109652,0.081543,0.1
1,8,1,dbmjbp674qds73dsd5i0,3.0,4096,0.284912,0.526176,0.530133,0.283633,0.080811,0.1
2,4,0,dbmjc2hkbkrc73cpju90,3.0,4096,0.632812,0.615371,0.612779,0.091049,0.057617,0.1
3,4,1,dbmjcghkbkrc73cpjv6g,3.0,4096,0.251465,0.591295,0.612779,0.227184,0.057129,0.1


,seed,keep,pruning_percent,parameters,reconstructed_batches,batch_size,quantum_failed_batches,quantum_mean_test_accuracy_successful_batches,random_mean_test_accuracy,quantum_accuracy_utility_zero_for_failed_batch,quantum_mean_validation_loss_successful_batches,random_mean_validation_loss,first_frozen_quantum_state,first_frozen_random_state,first_frozen_quantum_test_accuracy,first_frozen_random_test_accuracy,scope
0,1001,8,50.0,12730,128,32,0,0.826510,0.827212,0.826510,0.519269,0.518368,51795,8815,0.8242,0.8258,"Exploratory, first preselected initialization,..."
1,1001,4,75.0,6370,128,32,0,0.812299,0.813596,0.812299,0.557491,0.554388,18498,864,0.8171,0.8118,"Exploratory, first preselected initialization,..."
